In [ ]:
# Cell 1 -- Setup + download ap-matched-sentences.db from YOUR Google Drive
#
# BEFORE RUNNING:
#   1. Download ap-matched-sentences.db from the NewsEdits Drive folder:
#      https://drive.google.com/drive/folders/17a5S3liA0C91XbgnMBUQBo-NVb22Z9xf
#      (open 'matched-sentences' subfolder -> download ap-matched-sentences.db)
#   2. Upload that file to YOUR Google Drive
#   3. Right-click the file in YOUR Drive -> Get link -> Copy link
#      Link format: https://drive.google.com/file/d/FILE_ID/view?usp=sharing
#   4. Paste just the FILE_ID part into GDRIVE_FILE_ID below
#
# Requires: Wikipedia training checkpoint (wiki_model.pt)
#           Run kaggle_notebook.ipynb first if missing.

# ---- PASTE YOUR FILE ID HERE ----
GDRIVE_FILE_ID = 'PASTE_YOUR_FILE_ID_HERE'
# Example: '1A2B3C4D5E6F7G8H9I0J1K2L3M4N5O6P'
# ----------------------------------

!pip install transformers scikit-learn scipy gdown -q

import os, sys, glob
from pathlib import Path

# Clone / pull repo
REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')
sys.path.insert(0, str(REPO / 'delta_system'))
print('Repo OK')

# Show available checkpoints
ckpts = glob.glob('/kaggle/working/checkpoints/*.pt')
print('Checkpoints:', ckpts)

# Download the .db file from your Google Drive
DB_PATH = Path('/kaggle/working/newsedits/ap-matched-sentences.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    if GDRIVE_FILE_ID == 'PASTE_YOUR_FILE_ID_HERE':
        print()
        print('ERROR: You must paste your Google Drive file ID into GDRIVE_FILE_ID above.')
        print('Steps:')
        print('  1. Upload ap-matched-sentences.db to your Google Drive')
        print('  2. Right-click -> Get link -> copy it')
        print('  3. The FILE_ID is the long string between /d/ and /view in the link')
        print('  4. Paste it into GDRIVE_FILE_ID at the top of this cell')
    else:
        import gdown
        print(f'Downloading from your Google Drive (file ID: {GDRIVE_FILE_ID})...')
        gdown.download(id=GDRIVE_FILE_ID, output=str(DB_PATH), quiet=False)
else:
    print(f'Already exists: {DB_PATH}')

print()
print(f'DB_PATH exists: {DB_PATH.exists()}')
if DB_PATH.exists():
    print(f'File size: {DB_PATH.stat().st_size / 1e6:.1f} MB')

In [ ]:
# Cell 2 -- Run zero-shot NewsEdits evaluation
# Wikipedia-trained G evaluated on news revision pairs -- no news training at all

DB_PATH = '/kaggle/working/newsedits/ap-matched-sentences.db'

!python /kaggle/working/multihop-retrieval/delta_system/newsedits_zeroshot_eval.py \
    --db {DB_PATH} \
    --ckpt /kaggle/working/checkpoints/wiki_model.pt \
    --n 1000 \
    --min_added 2